# Explainable AI for Bridge Deck Crack Detection
### Comparative Evaluation of XAI Methods: Faithfulness Metrics + Expert Plausibility Rating

**Pipeline overview (TRB 2027 study):**
1. Fine-tune 3 lightweight classifiers (ResNet-18, MobileNetV3-Small, ViT-Tiny) on **SDNET2018 bridge deck** patches (cracked vs. non-cracked)
2. Generate explanations with **Grad-CAM++, Eigen-CAM, Score-CAM** (+ optional **SHAP** on CNNs)
3. **Quantitative faithfulness:** deletion / insertion curves (AUC)
4. **Localization vs. ground truth:** cross-dataset evaluation on **DeepCrack** pixel-level masks (pointing game + energy inside mask)
5. **Runtime** per method (deployment practicality)
6. Export **blinded expert-rating package** (overlays + rating template + hidden key)

**How to run:**
- Runtime > Change runtime type > **GPU (T4 is fine)**
- Run cells top to bottom
- First pass: leave `QUICK_TEST = True` to verify everything end-to-end (~10-15 min), then set `False` for the full run
- All results are saved under `/content/xai_results/` (download the zip at the end)


In [ ]:
# ============ 1. INSTALLS ============
!pip -q install grad-cam timm kagglehub shap opencv-python-headless scikit-learn pandas matplotlib


In [ ]:
# ============ 2. IMPORTS & CONFIG ============
import os, glob, json, random, shutil, time, zipfile, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights, mobilenet_v3_small, MobileNet_V3_Small_Weights
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from PIL import Image

from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM, ScoreCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
QUICK_TEST   = False    # True = tiny end-to-end sanity run; False = full experiment
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 1 if QUICK_TEST else 25
LR           = 3e-4          # peak LR for OneCycle
LABEL_SMOOTH = 0.05
USE_ALL_NEGATIVES = True     # keep all non-cracked images; balance via sampler instead of discarding data
NEG_POS_RATIO = 2.0    # undersample non-cracked to at most 2x cracked (SDNET is imbalanced ~1:5)
MAX_PER_CLASS = 300 if QUICK_TEST else None   # cap images per class in QUICK_TEST

N_FAITH      = 16 if QUICK_TEST else 100      # eval images for deletion/insertion
N_SCORECAM   = 8  if QUICK_TEST else 50       # Score-CAM is slow; evaluate on a subset
N_LOCAL      = 24 if QUICK_TEST else 150      # DeepCrack patches for localization
N_RATING_IMG = 10                              # images in the expert rating package
RUN_SHAP     = False   # set True to add SHAP (GradientExplainer, CNNs only, slow)
N_SHAP       = 8 if QUICK_TEST else 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

ROOT     = "/content"
OUT_DIR  = os.path.join(ROOT, "xai_results")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


## 3. Data: SDNET2018 (bridge deck subset)

Primary path: automatic download via `kagglehub` (public dataset, no token needed in Colab).
If that fails, download SDNET2018 manually from Utah State University
(https://digitalcommons.usu.edu/all_datasets/48/), upload/unzip it to `/content/`, and set `SDNET_ROOT` to that folder.
We use only the **deck** subset: `D/CD` (cracked) and `D/UD` (non-cracked).

In [ ]:
# ============ 3. DOWNLOAD SDNET2018 & LOCATE DECK FOLDERS ============
SDNET_ROOT = None
try:
    import kagglehub
    SDNET_ROOT = kagglehub.dataset_download("aniruddhsharma/structural-defects-network-concrete-crack-images")
    print("SDNET2018 downloaded to:", SDNET_ROOT)
except Exception as e:
    print("kagglehub download failed:", e)
    print("-> Download SDNET2018 manually, unzip to /content/SDNET2018, then set SDNET_ROOT below.")
    SDNET_ROOT = "/content/SDNET2018"   # adjust if needed

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def find_deck_dirs(root):
    """Locate cracked/non-cracked DECK image folders across known SDNET2018 layouts
    (USU original: D/CD, D/UD; Kaggle mirrors: Decks/Cracked, Decks/Non-cracked)."""
    cracked, uncracked = [], []
    for dirpath, _, filenames in os.walk(root):
        if not any(f.lower().endswith(IMG_EXT) for f in filenames):
            continue
        base   = os.path.basename(dirpath)
        parent = os.path.basename(os.path.dirname(dirpath)).lower()
        b = base.lower()
        deck_parent = parent in ("d", "decks", "deck") or "deck" in parent
        if base == "CD" or (deck_parent and "crack" in b and not b.startswith(("non", "un"))):
            cracked.append(dirpath)
        elif base == "UD" or (deck_parent and (b.startswith(("non", "un")) or "uncrack" in b)):
            uncracked.append(dirpath)
    return cracked, uncracked

cracked_dirs, uncracked_dirs = find_deck_dirs(SDNET_ROOT)
print("Cracked deck dirs:   ", cracked_dirs)
print("Non-cracked deck dirs:", uncracked_dirs)
assert cracked_dirs and uncracked_dirs, "Deck folders not found -- check SDNET_ROOT and folder layout."

def collect(dirs):
    files = []
    for d in dirs:
        files += [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(IMG_EXT)]
    return sorted(files)

pos_files = collect(cracked_dirs)     # label 1 = cracked
neg_files = collect(uncracked_dirs)   # label 0 = non-cracked
print(f"Found {len(pos_files)} cracked / {len(neg_files)} non-cracked deck patches")

# Undersample negatives for class balance; optional cap for QUICK_TEST
set_seed()
random.shuffle(neg_files)
if not USE_ALL_NEGATIVES:
    neg_files = neg_files[: int(len(pos_files) * NEG_POS_RATIO)]
if MAX_PER_CLASS:
    pos_files = pos_files[:MAX_PER_CLASS]
    neg_files = neg_files[:MAX_PER_CLASS]
print(f"Using {len(pos_files)} cracked / {len(neg_files)} non-cracked after balancing")


In [ ]:
# ============ 4. DATASET & SPLITS ============
all_files  = pos_files + neg_files
all_labels = [1]*len(pos_files) + [0]*len(neg_files)

idx = list(range(len(all_files)))
set_seed(); random.shuffle(idx)
n = len(idx)
train_idx = idx[: int(0.7*n)]
val_idx   = idx[int(0.7*n): int(0.85*n)]
test_idx  = idx[int(0.85*n):]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CrackDataset(Dataset):
    def __init__(self, files, labels, tf):
        self.files, self.labels, self.tf = files, labels, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        return self.tf(img), self.labels[i], self.files[i]

def subset(ix, tf): return CrackDataset([all_files[i] for i in ix], [all_labels[i] for i in ix], tf)

train_ds, val_ds, test_ds = subset(train_idx, train_tf), subset(val_idx, eval_tf), subset(test_idx, eval_tf)
# balanced batches without discarding data: weighted sampler over the (imbalanced) train split
train_labels_arr = np.array([all_labels[i] for i in train_idx])
class_counts = np.bincount(train_labels_arr, minlength=2)
sample_w = (1.0 / class_counts)[train_labels_arr]
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                num_samples=len(train_labels_arr), replacement=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
print("train class counts (0=non-cracked, 1=cracked):", class_counts)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")


## 5. Models & Training
Three lightweight architectures, all ImageNet-pretrained and fully fine-tuned:
ResNet-18 and MobileNetV3-Small (CNNs) plus ViT-Tiny (attention-based contrast).

In [ ]:
# ============ 5. MODEL FACTORY & TRAINING ============
def make_model(name):
    if name == "resnet18":
        m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "mobilenetv3":
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    elif name == "vit_tiny":
        m = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=2)
    else:
        raise ValueError(name)
    return m.to(DEVICE)

MODEL_NAMES = ["resnet18", "mobilenetv3", "vit_tiny"]

@torch.no_grad()
def get_probs(model, dl):
    model.eval(); ys, probs = [], []
    for x, y, _ in dl:
        p = F.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu().numpy()
        probs += list(p); ys += list(y.numpy())
    return np.array(ys), np.array(probs)

def metrics_at(ys, probs, thr):
    ps = (probs >= thr).astype(int)
    acc = accuracy_score(ys, ps)
    pr, rc, f1, _ = precision_recall_fscore_support(ys, ps, average="binary", zero_division=0)
    return acc, pr, rc, f1, confusion_matrix(ys, ps)

def best_threshold(ys, probs):
    thrs = np.linspace(0.05, 0.95, 91)
    f1s = [metrics_at(ys, probs, t)[3] for t in thrs]
    return float(thrs[int(np.argmax(f1s))])

def train_model(name):
    print(f"\n===== {name} =====")
    set_seed()
    model = make_model(name)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=len(train_dl), pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    best_f1, ckpt = -1, os.path.join(CKPT_DIR, f"{name}.pt")
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for x, y, _ in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * x.size(0)
        ys_v, probs_v = get_probs(model, val_dl)
        acc, pr, rc, f1, _ = metrics_at(ys_v, probs_v, 0.5)
        print(f"ep {ep+1}/{EPOCHS} loss {tot/len(train_ds):.4f} | val acc {acc:.3f} P {pr:.3f} R {rc:.3f} F1 {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1; torch.save(model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt))
    ys_v, probs_v = get_probs(model, val_dl)
    thr = best_threshold(ys_v, probs_v)                 # tuned on VAL only
    ys_t, probs_t = get_probs(model, test_dl)
    acc, pr, rc, f1, cm = metrics_at(ys_t, probs_t, thr)
    acc5, pr5, rc5, f15, _ = metrics_at(ys_t, probs_t, 0.5)
    print(f"TEST @0.5  acc {acc5:.3f} P {pr5:.3f} R {rc5:.3f} F1 {f15:.3f}")
    print(f"TEST @thr={thr:.2f}  acc {acc:.3f} P {pr:.3f} R {rc:.3f} F1 {f1:.3f}\n{cm}")
    return model, thr, {"model": name, "threshold": thr, "test_acc": acc, "precision": pr,
                        "recall": rc, "f1": f1, "test_acc_at_0.50": acc5, "f1_at_0.50": f15}

models, thresholds, perf_rows = {}, {}, []
for name in MODEL_NAMES:
    m, thr, row = train_model(name)
    models[name] = m; thresholds[name] = thr; perf_rows.append(row)

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(os.path.join(OUT_DIR, "model_performance.csv"), index=False)
perf_df


## 6. XAI: Grad-CAM++, Eigen-CAM, Score-CAM
Explanations are generated for the **cracked** class on test images the model predicts as cracked
(explaining a positive detection, which is the inspector-facing use case).

In [ ]:
# ============ 6. XAI SETUP & SALIENCY GENERATION ============
def vit_reshape_transform(tensor, h=14, w=14):
    res = tensor[:, 1:, :].reshape(tensor.size(0), h, w, tensor.size(2))
    return res.permute(0, 3, 1, 2)

def target_layers(model, name):
    if name == "resnet18":    return [model.layer4[-1]]
    if name == "mobilenetv3": return [model.features[-1]]
    if name == "vit_tiny":    return [model.blocks[-1].norm1]

CAM_CLASSES = {"gradcam++": GradCAMPlusPlus, "eigencam": EigenCAM, "scorecam": ScoreCAM}

def make_cam(method, model, name):
    kw = dict(model=model, target_layers=target_layers(model, name))
    if name == "vit_tiny":
        kw["reshape_transform"] = vit_reshape_transform
    return CAM_CLASSES[method](**kw)

def denorm(x):
    x = x.clone()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        x[c] = x[c] * s + m
    return x.clamp(0, 1)

# --- pick faithfulness eval set: test images predicted cracked by ALL models (shared, fair comparison)
def predicted_cracked(model, dl, thr):
    model.eval(); keep = []
    with torch.no_grad():
        for x, y, paths in dl:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu().numpy()
            keep += [p for p, pb, yy in zip(paths, prob, y) if pb >= thr and yy == 1]
    return set(keep)

shared = set.intersection(*[predicted_cracked(models[n], test_dl, thresholds[n]) for n in MODEL_NAMES])
shared = sorted(shared)[:N_FAITH]
print(f"Shared true-positive eval set: {len(shared)} images")
assert len(shared) >= 5, "Too few shared true positives -- increase data or epochs."

eval_x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in shared])
eval_rgb = np.stack([denorm(t).permute(1, 2, 0).numpy() for t in eval_x])

# --- generate saliency maps: sal[model][method] -> (N, H, W), plus runtime bookkeeping
sal, runtime_rows = {}, []
CAM_BS = 16
SCORECAM_BS = 1   # Score-CAM expands input by #channels internally; keep tiny to avoid CUDA OOM
for mname in MODEL_NAMES:
    sal[mname] = {}
    for method in CAM_CLASSES:
        n_imgs = min(N_SCORECAM, len(shared)) if method == "scorecam" else len(shared)
        bs = SCORECAM_BS if method == "scorecam" else CAM_BS
        maps, t0 = [], time.time()
        cam = make_cam(method, models[mname], mname)
        for i in range(0, n_imgs, bs):
            batch = eval_x[i:min(i + bs, n_imgs)].to(DEVICE)
            g = cam(input_tensor=batch, targets=[ClassifierOutputTarget(1)] * batch.size(0))
            maps.append(g)
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        del cam
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        maps = np.concatenate(maps, 0)
        assert len(maps) == n_imgs, (mname, method, len(maps), n_imgs)
        dt = (time.time() - t0) / n_imgs
        sal[mname][method] = maps
        runtime_rows.append(dict(model=mname, method=method, sec_per_image=round(dt, 4), n=n_imgs))
        print(f"{mname:12s} {method:10s} {maps.shape}  {dt:.3f} s/img")

runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(os.path.join(OUT_DIR, "xai_runtime.csv"), index=False)
runtime_df

## 7. Faithfulness: Deletion / Insertion curves
- **Deletion**: progressively remove the most-salient pixels (replace with blurred image); a faithful map makes the "cracked" probability drop fast (**lower AUC = better**).
- **Insertion**: progressively reveal the most-salient pixels onto a blurred image; a faithful map recovers the prediction fast (**higher AUC = better**).

In [ ]:
# ============ 7. DELETION / INSERTION ============
N_STEPS = 30
_trapz = getattr(np, "trapezoid", getattr(np, "trapz", None))  # numpy 2.x renamed trapz

@torch.no_grad()
def crack_prob(model, x):
    return F.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu().numpy()

def del_ins_auc(model, img_t, sal_map, mode):
    """img_t: (3,H,W) normalized tensor; sal_map: (H,W). Returns AUC of prob-vs-fraction curve."""
    H, W = sal_map.shape
    order = np.argsort(sal_map.flatten())[::-1]          # most salient first
    blur = transforms.functional.gaussian_blur(img_t.unsqueeze(0), 51, 20.0).squeeze(0)
    start = img_t.clone() if mode == "deletion" else blur.clone()
    fill  = blur if mode == "deletion" else img_t
    probs, cur = [], start.clone()
    step = max(1, len(order) // N_STEPS)
    batch, states = [], []
    for k in range(0, len(order) + 1, step):
        if k > 0:
            ids = order[k-step:k].copy()
            ys, xs = np.unravel_index(ids, (H, W))
            cur[:, ys, xs] = fill[:, ys, xs]
        states.append(cur.clone())
    xb = torch.stack(states)
    ps = []
    for i in range(0, len(xb), 64):
        ps.append(crack_prob(model, xb[i:i+64]))
    ps = np.concatenate(ps)
    return float(_trapz(ps, dx=1.0 / (len(ps) - 1)))

faith_rows = []
for mname in MODEL_NAMES:
    for method, maps in sal[mname].items():
        for i in range(len(maps)):
            d = del_ins_auc(models[mname], eval_x[i], maps[i], "deletion")
            ins = del_ins_auc(models[mname], eval_x[i], maps[i], "insertion")
            faith_rows.append(dict(model=mname, method=method, image=os.path.basename(shared[i]),
                                   deletion_auc=d, insertion_auc=ins))
faith_df = pd.DataFrame(faith_rows)
faith_df.to_csv(os.path.join(OUT_DIR, "faithfulness_per_image.csv"), index=False)
faith_summary = faith_df.groupby(["model", "method"])[["deletion_auc", "insertion_auc"]].agg(["mean", "std"]).round(4)
faith_summary.to_csv(os.path.join(OUT_DIR, "faithfulness_summary.csv"))
print(faith_summary)


## 8. Localization vs. ground truth (DeepCrack, cross-dataset)
Models trained on SDNET deck patches are applied to **DeepCrack** test images (pixel-level masks).
This doubles as a cross-dataset generalization check. Metrics per saliency map:
- **Pointing game**: is the single most-salient pixel inside the (dilated) crack mask?
- **Energy inside mask**: fraction of total saliency mass falling on crack pixels (dilated).

In [ ]:
# ============ 8. DEEPCRACK DOWNLOAD & PATCH EXTRACTION ============
DC_ZIP = "/content/DeepCrack.zip"
DC_DIR = "/content/DeepCrack_data"
if not os.path.exists(DC_DIR):
    if not os.path.exists(DC_ZIP):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/yhlleo/DeepCrack/master/dataset/DeepCrack.zip", DC_ZIP)
    with zipfile.ZipFile(DC_ZIP) as z:
        z.extractall(DC_DIR)

dc_imgs = sorted(glob.glob(os.path.join(DC_DIR, "**", "test_img", "*.jpg"), recursive=True))
print("DeepCrack test images:", len(dc_imgs))

def mask_path(img_path):
    return img_path.replace("test_img", "test_lab").rsplit(".", 1)[0] + ".png"

def extract_patches(img_paths, n_total, patch=IMG_SIZE, per_img=3):
    """Crop patch-size windows centered on random crack pixels; keep patches with enough crack mass."""
    out = []
    set_seed()
    for p in img_paths:
        mp = mask_path(p)
        if not os.path.exists(mp): continue
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if img is None or msk is None: continue
        H, W = msk.shape
        if H < patch or W < patch: continue
        ys, xs = np.where(msk > 127)
        if len(ys) < 50: continue
        picks = np.random.choice(len(ys), min(per_img * 4, len(ys)), replace=False)
        added = 0
        for j in picks:
            cy = int(np.clip(ys[j], patch // 2, H - patch // 2))
            cx = int(np.clip(xs[j], patch // 2, W - patch // 2))
            y0, x0 = cy - patch // 2, cx - patch // 2
            m = msk[y0:y0+patch, x0:x0+patch]
            if (m > 127).mean() < 0.005:   # require some crack presence
                continue
            out.append((img[y0:y0+patch, x0:x0+patch].copy(), (m > 127).astype(np.uint8)))
            added += 1
            if added >= per_img: break
        if len(out) >= n_total: break
    return out[:n_total]

patches = extract_patches(dc_imgs, N_LOCAL)
print("Localization patches:", len(patches))

loc_x = torch.stack([eval_tf(Image.fromarray(im)) for im, _ in patches])
loc_masks = np.stack([m for _, m in patches])
kernel = np.ones((11, 11), np.uint8)
loc_masks_dil = np.stack([cv2.dilate(m, kernel) for m in loc_masks])


In [ ]:
# ============ 9. LOCALIZATION METRICS ============
def pointing_hit(sal_map, mask_dil):
    y, x = np.unravel_index(np.argmax(sal_map), sal_map.shape)
    return float(mask_dil[y, x] > 0)

def energy_inside(sal_map, mask_dil):
    s = sal_map - sal_map.min()
    tot = s.sum()
    return float((s * (mask_dil > 0)).sum() / tot) if tot > 0 else 0.0

loc_rows = []
for mname in MODEL_NAMES:
    for method in CAM_CLASSES:
        n_imgs = min(N_SCORECAM, len(loc_x)) if method == "scorecam" else len(loc_x)
        bs = SCORECAM_BS if method == "scorecam" else CAM_BS
        cam = make_cam(method, models[mname], mname)
        base = 0
        for i in range(0, n_imgs, bs):
            batch = loc_x[i:min(i + bs, n_imgs)].to(DEVICE)
            g = cam(input_tensor=batch, targets=[ClassifierOutputTarget(1)] * batch.size(0))
            for j in range(g.shape[0]):
                idx = base + j
                loc_rows.append(dict(model=mname, method=method, idx=idx,
                                     pointing_hit=pointing_hit(g[j], loc_masks_dil[idx]),
                                     energy_inside=energy_inside(g[j], loc_masks_dil[idx])))
            base += g.shape[0]
            del g, batch
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        del cam
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
loc_df = pd.DataFrame(loc_rows)
loc_df.to_csv(os.path.join(OUT_DIR, "localization_per_patch.csv"), index=False)
loc_summary = loc_df.groupby(["model", "method"])[["pointing_hit", "energy_inside"]].mean().round(4)
loc_summary.to_csv(os.path.join(OUT_DIR, "localization_summary.csv"))
print(loc_summary)

## 10. Optional: SHAP (GradientExplainer, CNNs only)
Set `RUN_SHAP = True` in the config cell to include SHAP. It is slow and is evaluated on a
subset; SHAP maps are fed through the same deletion/insertion metrics for comparability.

In [ ]:
RUN_SHAP = True

In [ ]:
# ============ 10. SHAP (OPTIONAL) ============
if RUN_SHAP:
    import shap
    bg_x = torch.stack([train_ds[i][0] for i in range(min(24, len(train_ds)))]).to(DEVICE)
    shap_rows = []
    for mname in ["resnet18", "mobilenetv3"]:   # ViT + GradientExplainer is unreliable; skip
        model = models[mname].eval()
        explainer = shap.GradientExplainer(model, bg_x)
        n_imgs = min(N_SHAP, len(shared))
        xs = eval_x[:n_imgs].to(DEVICE)
        sv = explainer.shap_values(xs)                 # list per class or array
        sv_pos = sv[1] if isinstance(sv, list) else sv[..., 1]
        maps = np.abs(sv_pos).sum(1) if sv_pos.shape[1] == 3 else np.abs(sv_pos).sum(-1)
        # normalize each map to [0,1]
        maps = np.stack([(m - m.min()) / (m.max() - m.min() + 1e-8) for m in maps])
        sal[mname]["shap"] = maps
        for i in range(n_imgs):
            d  = del_ins_auc(model, eval_x[i], maps[i], "deletion")
            ins = del_ins_auc(model, eval_x[i], maps[i], "insertion")
            shap_rows.append(dict(model=mname, method="shap", image=os.path.basename(shared[i]),
                                  deletion_auc=d, insertion_auc=ins))
        print(f"SHAP done: {mname} ({n_imgs} images)")
    if shap_rows:
        pd.concat([faith_df, pd.DataFrame(shap_rows)]).to_csv(
            os.path.join(OUT_DIR, "faithfulness_per_image.csv"), index=False)
else:
    print("SHAP skipped (RUN_SHAP = False)")


## 11. Blinded expert-rating package
Creates `rating_package/`: one overlay PNG per (image x method x model) with **coded IDs**,
a `rating_template.csv` for the rater, and a separate hidden `KEY_do_not_open.csv`.
Rubric per item (1-5 unless noted): localization plausibility, verification usefulness,
misleading (yes/no), free-text comment.

In [ ]:
# ============ 11. EXPORT RATING PACKAGE ============
RATE_DIR = os.path.join(OUT_DIR, "rating_package")
os.makedirs(RATE_DIR, exist_ok=True)

rate_imgs = list(range(min(N_RATING_IMG, len(shared))))
items, key_rows = [], []
set_seed()
for mname in MODEL_NAMES:
    for method, maps in sal[mname].items():
        for i in rate_imgs:
            if i >= len(maps): continue
            overlay = show_cam_on_image(eval_rgb[i].astype(np.float32), maps[i], use_rgb=True)
            items.append((overlay, mname, method, os.path.basename(shared[i])))

order = list(range(len(items)))
random.shuffle(order)
for rank, j in enumerate(order):
    overlay, mname, method, src = items[j]
    code_id = f"ITEM_{rank:03d}"
    cv2.imwrite(os.path.join(RATE_DIR, f"{code_id}.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    key_rows.append(dict(item_id=code_id, model=mname, method=method, source_image=src))

pd.DataFrame(key_rows).to_csv(os.path.join(OUT_DIR, "KEY_do_not_open.csv"), index=False)
pd.DataFrame({
    "item_id": [r["item_id"] for r in sorted(key_rows, key=lambda r: r["item_id"])],
    "localization_plausibility_1to5": "",
    "verification_usefulness_1to5": "",
    "misleading_yes_no": "",
    "comments": "",
}).to_csv(os.path.join(RATE_DIR, "rating_template.csv"), index=False)
print(f"Rating package: {len(key_rows)} items -> {RATE_DIR}")


## 12. Results roll-up, plots, and download

In [ ]:
# ============ 12. SUMMARY PLOTS & ZIP ============
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
faith_m = faith_df.groupby(["model", "method"])[["deletion_auc", "insertion_auc"]].mean().reset_index()
for ax, col, title in [(axes[0], "deletion_auc", "Deletion AUC (lower = better)"),
                       (axes[1], "insertion_auc", "Insertion AUC (higher = better)")]:
    piv = faith_m.pivot(index="method", columns="model", values=col)
    piv.plot(kind="bar", ax=ax, rot=0); ax.set_title(title); ax.set_xlabel("")
piv = loc_df.groupby(["model", "method"])["energy_inside"].mean().reset_index().pivot(
    index="method", columns="model", values="energy_inside")
piv.plot(kind="bar", ax=axes[2], rot=0); axes[2].set_title("Energy inside crack mask (higher = better)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "summary_plots.png"), dpi=200)
plt.show()

# qualitative grid: first 4 eval images x methods for resnet18
n_show = min(4, len(shared))
methods_show = list(sal["resnet18"].keys())
fig, axes = plt.subplots(n_show, len(methods_show) + 1, figsize=(3 * (len(methods_show) + 1), 3 * n_show))
axes = np.atleast_2d(axes)
for r in range(n_show):
    axes[r, 0].imshow(eval_rgb[r]); axes[r, 0].set_ylabel(os.path.basename(shared[r])[:14], fontsize=7)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    if r == 0: axes[r, 0].set_title("input")
    for c, method in enumerate(methods_show):
        maps = sal["resnet18"][method]
        if r < len(maps):
            axes[r, c+1].imshow(show_cam_on_image(eval_rgb[r].astype(np.float32), maps[r], use_rgb=True))
        axes[r, c+1].set_xticks([]); axes[r, c+1].set_yticks([])
        if r == 0: axes[r, c+1].set_title(method)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "qualitative_grid_resnet18.png"), dpi=200)
plt.show()

shutil.make_archive("/content/xai_results_bundle", "zip", OUT_DIR)
print("\nAll results zipped -> /content/xai_results_bundle.zip  (download from the Files panel)")


## 13. After the rating session: inter-rater agreement (run later)
Once you (and optionally the PennDOT engineer) fill `rating_template.csv`, merge with the key and,
if two raters, compute weighted Cohen's kappa:

```python
from sklearn.metrics import cohen_kappa_score
r1 = pd.read_csv("rating_rater1.csv"); r2 = pd.read_csv("rating_rater2.csv")
key = pd.read_csv("KEY_do_not_open.csv")
m = r1.merge(r2, on="item_id", suffixes=("_r1", "_r2")).merge(key, on="item_id")
kappa = cohen_kappa_score(m["localization_plausibility_1to5_r1"],
                          m["localization_plausibility_1to5_r2"], weights="quadratic")
print("Weighted kappa:", kappa)
print(m.groupby("method")[["localization_plausibility_1to5_r1", "verification_usefulness_1to5_r1"]].mean())
```

**Paper checklist reminders:** report per-method mean +/- std for deletion, insertion, energy-inside,
pointing game, runtime; the divergence analysis (faithfulness rank vs. expert plausibility rank) is the
headline finding; note DeepCrack evaluation is cross-dataset (generalization framing).